# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


The PATH did not work so the cell returned the bite size of the table from StringIO

In [2]:
import pandas as pd

cat = pd.read_csv('/content/product_catalog.csv')

print(f"Loaded {len(cat):,} products")
cat.head()


Loaded 45 products


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


I added the csv to the notebook enviroment and then checked to make sure all 45 products actually loaded.

### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# TODO (hint: .str.lower().str.contains('pop'))

looks_pop = cat [
    cat['product_name'].str.lower().str.contains('pop', case = False, na = False)
]

print(f"Found {len(looks_pop):,} products that look like Pop-Tarts")
looks_pop.head()

Found 4 products that look like Pop-Tarts


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


I used the hint to find all products that contained the lower case string 'pop', a part of the name of all products that have a good chance of being poptarts.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# TODO

is_strawberry = looks_pop['product_name'].str.lower().str.contains('strawberry', case = False, na = False)

strawberry_products = looks_pop[is_strawberry]
other_products = looks_pop[~is_strawberry]

print("Possibly Strawberry - check there are Poptarts")
display(strawberry_products [['sku', 'product_name']])

print("Other matches - keep other matches separate")
display(other_products [['sku', 'product_name']])

Possibly Strawberry - check there are Poptarts


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry


Other matches - keep other matches separate


,sku,product_name


I create two print displays to show the sku names I believe mean strawberry poptart, and to show which non strawberry poptart rows were found. I found four products, all with difference product names (this is super messy) that represent strawberry poptarts.

**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
# TODO
rows_before_mapping = len(cat)


sku_map = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].replace(sku_map)

print(f"Rows before mapping: {rows_before_mapping:,}")

display(cat[['sku', 'canonical_sku', 'product_name']])

Rows before mapping: 45


,sku,canonical_sku,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry
3,PT_STRAWBERRY,PT_STRAWBERRY,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack
6,WTR_24PK,WTR_24PK,Water - 24 Pack
7,BATT-100,BATT-100,BATTERIES AA 4PK
8,BAT-AA4,BAT-AA4,AA Batteries 4-Pack
9,BATT_AA_4PK,BATT_AA_4PK,AA Batteries 4-Pack


I created a SKU map to make all of the poptart items map to the same product, PT-12.  then made canonical_sku to save the product addresses, but without deleting the other products that map to other addresses.  

I also went back to add 'rows_before_mapping' so I could confirm there were 45 rows.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
# TODO

print("Before normalization")
display (cat['category'].value_counts(dropna = False))

cat['category_clean'] = cat['category'].str.strip().str.title()

print("After normalization")
display (cat['category_clean'].value_counts(dropna = False))

Before normalization


,count
category,
Emergency,5
Emergency Supplies,3
EMERGENCY,3
ENTERTAINMENT,3
GROCERY,3
Breakfast,2
entertainment,2
household,2
ELECTRONICS,2


After normalization


,count
category_clean,
Emergency,9
Entertainment,5
Grocery,5
Electronics,4
Household,4
Emergency Supplies,3
Breakfast,3
Baby,3
Snacks,2


I cleaned the data by removing the spaces at the begining of title and captializing all first letters so that category with multiple titlesthat all represented the same category displayed as pointing to the same thing.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# TODO

clean_names = (
    cat['product_name']
    .fillna('')
    .str.lower()
    .str.replace(r'[^\w\s]', ' ', regex = True)
)

cat['name_key'] = clean_names.apply(
    lambda name: ''.join(sorted(name.split()))
)

candidates = cat[cat['name_key'] != ''].groupby('name_key').filter(
    lambda group: group['sku'].nunique() > 1
)

display(
    candidates[
        ['name_key', 'sku', 'product_name', 'package_size']
    ].sort_values(['name_key', 'sku'])
)

,name_key,sku,product_name,package_size
12,24beerlightpack,BEER-24,Light Beer (24-pack),bundle
13,24beerlightpack,LTBEER24,Light Beer (24-pack),bundle
8,4aabatteriespack,BAT-AA4,AA Batteries 4-Pack,4pk
9,4aabatteriespack,BATT_AA_4PK,AA Batteries 4-Pack,24 pack
29,aidfirstkit,FAID-01,First-Aid Kit,4 pack
30,aidfirstkit,FIRSTAID-KIT,First Aid Kit,4 pack
14,boardgamemonopoly,BG-MONO,Monopoly Board Game,1pk
16,boardgamemonopoly,GAME-MONO,Board Game - Monopoly,1pk
15,boardgamemonopoly,MONOPOLY-01,Monopoly Board Game,24-pack
23,breadwhite,BREAD-WH,WHITE BREAD,24-pack


I created a clean 'name_key' to display the products to make all possible duplicates appear next to one another. I followed the instructions and normalized by making everything lowercase, filling nas, replacing faulty punctuation, and sort words in names, alphabetically. Making 'canidates' I then attempted to find name keys that represented multiple skus.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO

assert cat['canonical_sku'].notna().all(), \
    "Some conical SKUs missing"

assert len(cat) == rows_before_mapping, \
    "The number of rows changed"

print("Checks passed: no missing cononical SKUs and no change in row number")



Checks passed: no missing cononical SKUs and no change in row number


Created assert statement to make sure that in my cleaning I did not accidently remove cononical SKUs or add/remove rows.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A. I noticed that light beer and white bread both appear to be fragmented. Light beer has two different SKUs, 'BEER-24' and 'LTBEER24' that both have the same name and package size. I would group them under LTBEER24, so the SKU notes that they are light beer compared to other types of beer. White bread has a similar fragmenting error as there are two different SKUs, 'BREAD-WH' and 'WHITEBREAD 24' that both represent a 24 pack fo white bread with the product name 'WHITE BREAD'. I would group them under 'WHITEBREAD 24' to distinguish them from other sized packs of bread.



B. Consolidation stops being a data cleaning task and becomes a business decison when there are items that look the same numerically but may come from different distributors leading one item to be more popular than the other. Canned soup could be an example of this where there are two different SKUs that appear to point to the same item but it is possible a supply chain manager intentionally wants them labeled differently so he/she can distuigh which sells better. In a case of this matter, I would want to talk to the supply chain manager to figure out if there is a reason why the two near identical items have different SKUs.